### 🏷️ **Credits & License**

* 🔗 [MOSS-TTS GitHub Repository](https://github.com/OpenMOSS/MOSS-TTS)
* 🤗 [MOSS-TTS-v1.5 (8B) on Hugging Face](https://huggingface.co/OpenMOSS-Team/MOSS-TTS-v1.5)
* 📄 **License**: Provided under the [Apache License 2.0](https://github.com/OpenMOSS/MOSS-TTS/blob/main/LICENSE)

### ⚙️ **Runtime**

**Runtime → Change runtime type → A100 GPU.** The 8B model plus its audio codec need about 25 GB of GPU memory, so T4 and L4 are too small.

### ⚠️ **Usage Disclaimer**

Use of this voice cloning model is subject to strict ethical and legal standards. By using this tool, you agree **not to** engage in any of the following prohibited activities:

* **Fraud or Deception**: Using cloned voices to create misleading or fraudulent content.
* **Impersonation**: Replicating someone's voice without their explicit permission, especially for malicious, harmful, or deceptive purposes.
* **Illegal Activities**: Employing the model in any manner that violates local, national, or international laws and regulations.
* **Harmful Content Generation**: Creating offensive, defamatory, or unethical material, including content that spreads misinformation or causes harm.

> ⚖️ **Legal Responsibility**
> The developers of this tool disclaim all liability for misuse. **Users bear full responsibility** for ensuring that their usage complies with all applicable laws, regulations, and ethical guidelines.

In [ ]:
#@title Install MOSS-TTS
import subprocess, time

# Query the GPU with nvidia-smi instead of torch, so this notebook kernel never holds GPU memory itself.
try:
    _row = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                          capture_output=True, text=True, check=True).stdout.strip().splitlines()[0]
    _gpu_name, _gpu_mib = [part.strip() for part in _row.split(",")]
except Exception:
    raise SystemExit("❌ No GPU. Runtime → Change runtime type → A100 GPU, then run this cell again.")
if int(_gpu_mib) / 1024 < 30:
    raise SystemExit(f"❌ {_gpu_name} has {int(_gpu_mib) / 1024:.0f} GB. The 8B model needs an A100 (40 GB): Runtime → Change runtime type → A100 GPU.")

# A previous app.py still running would keep ~24 GB of GPU memory and block the next start.
if subprocess.run(["pkill", "-f", "app.py"]).returncode == 0:
    print("🛑 Stopped a previous MOSS-TTS app that was still running")
    time.sleep(5)

%cd /content/
!rm -rf ./Moss-8B
!git clone https://github.com/hirannalaka19/Moss-8B.git
%cd ./Moss-8B
!pip install -r colab.txt
from IPython.display import clear_output
clear_output()
print(f"✅ Installation complete. GPU: {_gpu_name}")

In [ ]:
#@title Run Gradio APP
#@markdown Uses the `HF_TOKEN` saved in Colab **Secrets** (🔑 icon in the left sidebar): add a secret named `HF_TOKEN` with your Hugging Face token (create one at https://huggingface.co/settings/tokens, type: **Read**) and enable **Notebook access**. One-time setup — it persists across all sessions.
#@markdown
#@markdown The first run downloads ~24 GB of model weights (about 5–10 minutes). Then open the public `*.gradio.live` link.

import os, subprocess, sys, time

HF_TOKEN = ""
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or ""
except Exception:
    HF_TOKEN = ""

if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    print("🔑 Using HF_TOKEN from Colab Secrets — full download speed")
else:
    print("⚠️ No HF_TOKEN secret found — add it via the 🔑 Secrets sidebar (name: HF_TOKEN) and enable Notebook access. Anonymous downloads are throttled.")

MODEL_REPOS = ["OpenMOSS-Team/MOSS-TTS-v1.5", "OpenMOSS-Team/MOSS-Audio-Tokenizer"]

# Hugging Face's CDN intermittently rejects its own signed URLs with
# 403 "invalid key pair id" (a server-side glitch). Already-downloaded files
# are cached, so retrying only fetches what is still missing — we retry the
# fast HTTP path several times, then fall back to the Xet protocol path.
def try_download(disable_xet):
    env = dict(os.environ)
    if disable_xet:
        env["HF_HUB_DISABLE_XET"] = "1"
    else:
        env.pop("HF_HUB_DISABLE_XET", None)
    script = "from huggingface_hub import snapshot_download; [snapshot_download(r) for r in %r]" % (MODEL_REPOS,)
    return subprocess.call([sys.executable, "-c", script], env=env) == 0

ok = False
for attempt in range(1, 5):
    print(f"⬇️ Downloading the model — fast HTTP path (attempt {attempt}/4) ...")
    if try_download(disable_xet=True):
        ok = True
        break
    print("⚠️ Hit the Hugging Face CDN glitch — retrying (already-downloaded files are kept) ...")
    time.sleep(5)

if not ok:
    for attempt in range(1, 3):
        print(f"⬇️ Falling back to the Xet protocol path (attempt {attempt}/2, can be slower) ...")
        if try_download(disable_xet=False):
            ok = True
            break
        time.sleep(5)

if not ok:
    raise SystemExit("❌ All download attempts failed — Hugging Face may be having an outage (check https://status.huggingface.co). Re-run this cell to retry.")

print("✅ Model downloaded")

# Model is now cached locally, so the app starts without re-downloading.
os.environ["HF_HUB_DISABLE_XET"] = "1"

# Only one copy of the 8B model fits on the GPU: stop an app left running by an earlier run of this cell.
if subprocess.run(["pkill", "-f", "app.py"]).returncode == 0:
    print("🛑 Stopped the previous MOSS-TTS app (it was still holding GPU memory)")
    time.sleep(5)
_used_mib = int(subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                               capture_output=True, text=True).stdout.strip().splitlines()[0] or 0)
if _used_mib > 4096:
    raise SystemExit(f"❌ {_used_mib / 1024:.0f} GB of GPU memory is still in use by something else. Runtime → Restart session, then run both cells again.")

%cd /content/Moss-8B
!python app.py